# Error Analysis: LightGBM v3 on test set

Goal: before writing up v1, look at what the model gets wrong and see if there's a pattern we missed. Three possible outcomes:

1. **Errors look random** — validates the model is doing its best within the feature constraints. Ship as-is.
2. **Clear pattern in errors** — points to a missing feature or data source, either to add now or note as a v2 direction.
3. **Systematic bias** — the model is broken in some specific way (e.g., always fails for injectables, always fails for new manufacturers). Would need fixing before ship.

Investigations:

- **Q1**: False negatives — confident 'no shortage' predictions that turned out positive. Most informative group.
- **Q2**: False positives — confident 'shortage' predictions that turned out negative.
- **Q3**: Error rate by subgroup (ATC, form, manufacturer size, drug age).
- **Q4**: What feature state predicts missed positives? (If a prior-shortage drug has the model confidently missing it, something's wrong.)
- **Q5**: Concentration — are errors dominated by a few manufacturers or molecule groups? If so, maybe domain-specific follow-ups.

## Setup: load data and retrain LightGBM on the current splits

We reuse `data_loader.py` for the splits and retrain inline for reproducibility. Training takes ~1 minute.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightgbm as lgb

from data_loader import (
    load_splits,
    TARGET,
    FEATURES,
    CATEGORICAL_FEATURES,
)

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)

train, val, test = load_splits(verbose=True)

In [ ]:
# Retrain the LightGBM baseline to produce test predictions.
# Same config as baseline.py so results match.

def prep(df):
    X = df[FEATURES].copy()
    for col in CATEGORICAL_FEATURES:
        X[col] = X[col].astype('category')
    return X

X_train, X_val, X_test = prep(train), prep(val), prep(test)
y_train, y_val, y_test = train[TARGET], val[TARGET], test[TARGET]

gbm = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=63,
    min_child_samples=100,
    random_state=42,
    verbose=-1,
)

gbm.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='average_precision',
    callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)],
    categorical_feature=CATEGORICAL_FEATURES,
)

print(f"Best iteration: {gbm.best_iteration_}")

# Score test set and attach to a working DataFrame for analysis.
test_scores = gbm.predict_proba(X_test)[:, 1]

df = test.copy()
df['score'] = test_scores
# Decile of score within the test set, useful for stratification.
df['score_decile'] = pd.qcut(df['score'], q=10, labels=False, duplicates='drop')

print(f"\nTest rows: {len(df):,}")
print(f"Positive rate: {df[TARGET].mean():.4f}")
print(f"Score range: {df['score'].min():.4f} to {df['score'].max():.4f}")

## Q1. False negatives — confidently-missed positives

These are the most operationally important errors: a shortage happened, the model predicted low risk, the operator didn't look. What do these cases have in common?

In [ ]:
# Rank positives by lowest score — the cases the model was most wrong about.
false_negatives = df[df[TARGET] == 1].sort_values('score').copy()

# Split by confidence: very low (bottom score decile among positives)
# vs moderate (middle deciles). Very-low-score positives are the model's
# worst misses.
print(f"Total positives in test: {len(false_negatives):,}")
print(f"Positives scored in bottom decile overall: "
      f"{(false_negatives['score_decile'] == 0).sum():,}")
print(f"Positives scored in bottom 3 deciles overall: "
      f"{(false_negatives['score_decile'] <= 2).sum():,}")
print(f"Positives scored in top decile overall: "
      f"{(false_negatives['score_decile'] == 9).sum():,}  "
      f"(model caught these)")

In [ ]:
# Most-confidently-missed positives: score very low.
worst_misses = false_negatives.head(50)

summary_cols = [
    'observation_date', 'din', 'score',
    'was_ever_in_shortage', 'shortages_prior_12m',
    'mfr_shortage_rate_12m', 'mfr_shortage_rate_3m',
    'peer_any_in_shortage_now_same_ai_group',
    'atc_anatomic_group', 'primary_form',
    'drug_age_years',
]
print("Top 20 most-confidently-missed positives:")
print(worst_misses[summary_cols].head(20).to_string(index=False))

In [ ]:
# Aggregate profile of confidently-missed positives vs overall positives.
positives = df[df[TARGET] == 1]
missed  = false_negatives.head(100)  # top 100 worst misses

profile = pd.DataFrame({
    'all_positives':      positives[summary_cols[3:]].select_dtypes('number').mean(),
    'worst_100_missed':   missed[summary_cols[3:]].select_dtypes('number').mean(),
})
profile['ratio_missed_vs_all'] = profile['worst_100_missed'] / profile['all_positives'].replace(0, np.nan)
print(profile)

In [ ]:
# Were ever in shortage?
print("Was ever in shortage (all positives):",
      f"{positives['was_ever_in_shortage'].mean():.3f}")
print("Was ever in shortage (worst 100 missed):",
      f"{missed['was_ever_in_shortage'].mean():.3f}")
print()

# ATC distribution of missed
print("ATC group distribution:")
comparison = pd.DataFrame({
    'all_positives':    positives['atc_anatomic_group'].value_counts(normalize=True),
    'worst_100_missed': missed['atc_anatomic_group'].value_counts(normalize=True),
}).fillna(0).sort_values('all_positives', ascending=False)
print(comparison.head(10))

**DECISION — Q1**:

Look at the profile table above. If worst-missed positives are cold-start drugs (`was_ever_in_shortage` low, `shortages_prior_12m` low) with low manufacturer signal, that's *expected* weakness — the features have nothing to work with. That's a documentable limitation, not a bug.

If worst-missed positives have *high* prior shortage signals or high manufacturer risk, something is wrong — the model is not using features it should be using.

## Q2. False positives — confident but wrong

These are drugs the model flagged as high-risk but no shortage occurred. They're operationally less serious than false negatives (just wasted operator attention) but patterns here can reveal model over-reliance on specific features.

In [ ]:
# Top-100 predictions that turned out negative
top_predictions = df.sort_values('score', ascending=False).head(100)

print(f"Top 100 predictions overall:")
print(f"  Actually positive: {top_predictions[TARGET].sum()} (= Precision@100)")
print(f"  Actually negative (false positives): {(~top_predictions[TARGET].astype(bool)).sum()}")
print()

# Of the high-confidence false positives, what do they look like?
fp = top_predictions[top_predictions[TARGET] == 0].copy()
print("Profile of confident false positives (top-100 with label 0):")
print(fp[summary_cols].head(10).to_string(index=False))

In [ ]:
# Are the false positives clustered on specific manufacturers?
# (Proxy: manufacturer represented via mfr_shortage_rate_12m, since we don't
# carry company_code through; use portfolio size as a group proxy.)
print("High-confidence top-1000 false-positive profile:")
top_1000 = df.sort_values('score', ascending=False).head(1000)
fp_1000 = top_1000[top_1000[TARGET] == 0]

print(f"  Count: {len(fp_1000)}")
print(f"  Median mfr_shortage_rate_12m: {fp_1000['mfr_shortage_rate_12m'].median():.3f}")
print(f"  Median mfr_shortage_rate_3m:  {fp_1000['mfr_shortage_rate_3m'].median():.3f}")
print(f"  Median shortages_prior_12m:   {fp_1000['shortages_prior_12m'].median():.1f}")
print()
print("  ATC distribution:")
print(fp_1000['atc_anatomic_group'].value_counts(normalize=True).head(5))

**DECISION — Q2**:

If false positives have very high shortage history and manufacturer rate features, the model is doing what it's supposed to — those drugs *were* at elevated risk but a shortage just didn't happen this 90-day window. That's noise, not a bug.

If false positives are concentrated on specific ATC groups or drugs that recently recovered, that would hint at a missing 'recovery' feature.

## Q3. Error rate by subgroup

Does the model fail worse on any specific subgroup beyond what we've already stratified?

In [ ]:
from sklearn.metrics import average_precision_score

def group_metrics(group_col, min_n=500):
    """PR-AUC by group, dropping tiny groups."""
    rows = []
    for grp, sub in df.groupby(group_col, observed=True):
        if len(sub) < min_n or sub[TARGET].sum() < 5:
            continue
        pr_auc = average_precision_score(sub[TARGET], sub['score'])
        rows.append({
            group_col: grp,
            'n': len(sub),
            'n_pos': int(sub[TARGET].sum()),
            'base_rate': sub[TARGET].mean(),
            'pr_auc': pr_auc,
            'lift_vs_base': pr_auc / sub[TARGET].mean(),
        })
    return pd.DataFrame(rows).sort_values('pr_auc', ascending=False)

print("PR-AUC by primary_form (top 10 forms by volume):")
form_metrics = group_metrics('primary_form', min_n=1000)
print(form_metrics.head(10).to_string(index=False))

In [ ]:
print("PR-AUC by primary_route:")
route_metrics = group_metrics('primary_route', min_n=1000)
print(route_metrics.head(10).to_string(index=False))

In [ ]:
print("PR-AUC by schedule:")
schedule_metrics = group_metrics('schedule', min_n=500)
print(schedule_metrics.to_string(index=False))

In [ ]:
# Drug age bucketed — is the model biased against new or old drugs?
df['drug_age_bucket'] = pd.cut(
    df['drug_age_years'],
    bins=[-0.01, 2, 5, 10, 20, 200],
    labels=['0-2y', '2-5y', '5-10y', '10-20y', '20y+'],
)

print("PR-AUC by drug age bucket:")
age_metrics = group_metrics('drug_age_bucket', min_n=1000)
print(age_metrics.to_string(index=False))

In [ ]:
# Manufacturer size — does the model work differently for boutique vs large mfrs?
df['mfr_size_bucket'] = pd.cut(
    df['mfr_portfolio_size'],
    bins=[-1, 10, 50, 200, 2000],
    labels=['tiny (<=10)', 'small (11-50)', 'medium (51-200)', 'large (200+)'],
)

print("PR-AUC by manufacturer portfolio size:")
mfr_metrics = group_metrics('mfr_size_bucket', min_n=1000)
print(mfr_metrics.to_string(index=False))

**DECISION — Q3**:

Flag any subgroup where:
- PR-AUC is much lower than the overall 0.135, AND
- The subgroup has meaningful positive count (otherwise it's noise)

These are candidates for a 'known weaknesses' section in the writeup.

## Q4. What feature state predicts missed positives?

Compare feature distributions between positives caught (top-decile score) vs missed (bottom-3-decile score).

In [ ]:
positives_caught = positives[positives['score_decile'] == 9]
positives_missed = positives[positives['score_decile'] <= 2]

print(f"Positives caught (top score decile): {len(positives_caught):,}")
print(f"Positives missed (bottom 3 deciles): {len(positives_missed):,}")
print()

key_features = [
    'was_ever_in_shortage',
    'shortages_prior_12m',
    'shortages_prior_36m',
    'shortages_all_prior',
    'mfr_shortage_rate_12m',
    'mfr_shortage_rate_3m',
    'mfr_shortage_rate_delta_3m_vs_12m',
    'mfr_portfolio_size',
    'mfr_discontinuation_rate_12m',
    'peer_any_in_shortage_now_same_ai_group',
    'peer_shortages_prior_12m_same_ai_group',
    'peer_discontinuations_prior_12m',
    'drug_age_years',
]

comparison = pd.DataFrame({
    'caught_mean': positives_caught[key_features].mean(),
    'missed_mean': positives_missed[key_features].mean(),
})
comparison['ratio'] = comparison['missed_mean'] / comparison['caught_mean'].replace(0, np.nan)
print("Feature means: caught vs missed positives")
print(comparison.sort_values('ratio').to_string())

**DECISION — Q4**:

Features where caught/missed ratio is very different reveal what the model relies on. Features with ratio close to 1 reveal what the model *isn't* using that could help distinguish these cases.

If missed positives systematically have low `was_ever_in_shortage`, low `shortages_prior_*`, and low `mfr_shortage_rate_*`, the story is clear: the model is good at catching drugs with visible prior trouble and bad at catching drugs with no warning signs. That's a data limitation, not a model limitation.

## Q5. Error concentration — are failures clustered?

In [ ]:
# Are errors concentrated on specific DINs? If a DIN appears as a missed
# positive repeatedly across months, it may warrant a DIN-specific note.
missed_by_din = (
    positives_missed.groupby('din')
    .size()
    .sort_values(ascending=False)
)
print(f"Unique DINs appearing among bottom-3-decile positives: {len(missed_by_din)}")
print(f"DINs with 2+ missed positives: {(missed_by_din >= 2).sum()}")
print(f"DINs with 4+ missed positives: {(missed_by_din >= 4).sum()}")
print()
print("Top 20 most-missed DINs:")
print(missed_by_din.head(20))

In [ ]:
# How many missed positives come from DINs with any prior-shortage history?
print("Of missed positives:")
print(f"  with was_ever_in_shortage=True: "
      f"{(positives_missed['was_ever_in_shortage'] == True).sum():,} "
      f"({positives_missed['was_ever_in_shortage'].mean():.1%})")
print(f"  with shortages_prior_12m > 0: "
      f"{(positives_missed['shortages_prior_12m'] > 0).sum():,}")
print(f"  cold-start (no prior shortage at all): "
      f"{(positives_missed['was_ever_in_shortage'] == False).sum():,}")

## Summary — what did we learn?

Fill this in after running. A template to force a conclusion:

| Question | Finding | Action |
|---|---|---|
| Q1 worst misses profile | mostly cold-start? / warm-start? | document as limitation OR investigate |
| Q2 confident FP profile | high-risk drugs that didn't trigger? | noise, not a bug |
| Q3 weakest subgroup | which one? | note in writeup |
| Q4 features not being used | any surprises? | possible new feature or note |
| Q5 error concentration | few DINs or many? | domain note if concentrated |

**Likely outcome based on v3 architecture**: errors are concentrated in cold-start drugs with no prior history. The features can't predict what has no precedent. This would be a clean portfolio limitation to document rather than a problem to fix.

**Alternative outcome**: if there's something unexpected — a specific ATC group that fails, a feature that should be helping but isn't, an error cluster — we have one more actionable insight before shipping.